# Inference — Training-Time Interpretability

Run inference locally on a checkpoint downloaded from the Modal Volume.

## 1. Download a checkpoint

First, see what checkpoints are available:
```bash
modal volume ls tti-checkpoints
modal volume ls tti-checkpoints expt1_baseline_seed42
```

Then download one:
```bash
modal volume get tti-checkpoints expt1_baseline_seed42/step_5000/checkpoint.pt checkpoints/expt1_baseline_step5000.pt
```

Set `CHECKPOINT_PATH` below to wherever you saved it.

In [1]:
import os
import sys
sys.path.insert(0, "..")

# Suppress HuggingFace progress bars (avoids ipywidgets rendering errors)
os.environ["TOKENIZERS_PARALLELISM"] = "false"
import transformers
transformers.logging.set_verbosity_error()

import torch
from transformers import AutoTokenizer
from src.model import GPT, GPTConfig

CHECKPOINT_PATH = "../checkpoints/expt1_baseline_step3000.pt"  # update this

device = (
    torch.device("mps") if torch.backends.mps.is_available()
    else torch.device("cpu")
)
print(f"Using device: {device}")

Using device: mps


In [2]:
# Load checkpoint
ckpt = torch.load(CHECKPOINT_PATH, map_location=device, weights_only=True)
print(f"Checkpoint from step {ckpt['step']}")

# Build model with same config used during training
config = GPTConfig(
    vocab_size=50257,
    max_seq_len=512,
    n_layers=6,
    n_heads=6,
    d_model=384,
    d_ff=1536,
    dropout=0.0,  # no dropout at inference
    bias=False,
)
model = GPT(config).to(device)
model.load_state_dict(ckpt["model"])
model.eval()
print(f"Loaded model ({model.num_params():,} params)")

tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

Checkpoint from step 5000
Loaded model (30,122,112 params)


In [3]:
@torch.no_grad()
def generate(
    prompt: str,
    max_new_tokens: int = 200,
    temperature: float = 0.8,
    top_k: int = 50,
) -> str:
    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)

    for _ in range(max_new_tokens):
        # Crop to max_seq_len if needed
        context = input_ids[:, -config.max_seq_len:]
        logits, _, _ = model(context)
        logits = logits[:, -1, :]  # last token

        # Temperature scaling
        logits = logits / temperature

        # Top-k filtering
        if top_k > 0:
            v, _ = torch.topk(logits, top_k)
            logits[logits < v[:, -1:]] = float("-inf")

        probs = torch.softmax(logits, dim=-1)
        next_token = torch.multinomial(probs, num_samples=1)
        input_ids = torch.cat([input_ids, next_token], dim=1)

        if next_token.item() == tokenizer.eos_token_id:
            break

    return tokenizer.decode(input_ids[0], skip_special_tokens=True)

In [4]:
prompt = "Once upon a time, there was a little girl named"
output = generate(prompt, max_new_tokens=200, temperature=0.8, top_k=50)
print(output)

Once upon a time, there was a little girl named Lily. She loved to sing and dance. One day, she flew over to her mommy and said, "Mommy, I want to sing too!" 

Her mommy said, "Lily, it's time to go home. We need to take a nap." 

Lily said, "But mommy, I want to go to bed. Can you please come with me?" 

Her mommy replied, "Sure, honey. I'll take you to the kitchen and get you a nap." 

Lily started to sing and dance until she fell asleep. When she woke up, she saw that her mommy had passed her to the kitchen. She asked, "Are you okay, my mommy?" 

Her mommy said, "I'm okay, Lily. I still had fun at bed and dancing." 

Lily felt happy that she helped her mommy and they went to bed together. 


In [5]:
# Try a few prompts side by side
prompts = [
    "Once upon a time",
    "Tom and his dog went to",
    "The little dragon was sad because",
]

for p in prompts:
    print(f"--- Prompt: {p!r} ---")
    print(generate(p, max_new_tokens=150, temperature=0.9))
    print()

--- Prompt: 'Once upon a time' ---
Once upon a time, there was a little girl named Lily. She loved to play outside with her dog, Max. One day, Lily went to the park to play with Max. They saw a big storm coming. Max was scared and tried to run away, but the storm was too fast. 

Lily and Max went back home and saw a small tree on the ground. The tree had a hole in it. It was an airplane. Max said, "Wow, Lily! You're not scared anymore." 

They started to play with Max and they had a good time. They made funny noises and laughed. The rain was all over it. From that day on, Max always made sure to protect Lily from any storm. And Lily

--- Prompt: 'Tom and his dog went to' ---
Tom and his dog went to the park. As they walked, they saw a big yellow tree. The tree had a long trunk that looked like a fairy. A fairy had a big smile.

The fairy saw Tom and his dog. He flew down to the tree and showed them to the tree. He said, "Look at the tree! He has a magic wand that will make you fly. He 